# Notebook 03 — Huấn Luyện Các Mô Hình Non-SA (Direct & Residual) & Walk-Forward

**Đồ án môn học:** IS403 — Phân Tích Dữ Liệu Kinh Doanh (Module 1: Dự báo giá đồng 1 phiên trước)  
**Tác giả:** Viên Đăng Khoa  
**Mục tiêu notebook:** Đồng khớp 100% giao thức tiền xử lý, chia tập và Walk-Forward với Notebook 02. Định nghĩa tách từng cell cho 5 họ kiến trúc (`LSTM`, `GRU`, `Causal CNN-LSTM`, `Causal CNN-GRU` trên cả hai nhánh **Direct** và **Residual**, cùng `Ridge Residual`), huấn luyện đầy đủ trên **4 bộ đặc trưng có Copper (`FS01`, `FS02`, `FS04`, `FS06`) × 4 tỷ lệ chia × 3 chiến lược** (432 lượt fit Holdout, lưu trọn bộ trọng số `.keras`/`.joblib` và `scaler`) + **Walk-Forward căn chỉnh 80/20** (542 phiên, 9 folds × 5 seeds = 369 fits).

### Mục lục
1. Cấu hình đồng nhất với Notebook 02 & nạp dữ liệu gốc
2. Chia tập chống rò rỉ (`make_context`) & trọng số suy giảm (`sample_weights`)
3. Định nghĩa tách từng cell cho từng họ mô hình (`LSTM`, `GRU`, `CNN-LSTM`, `CNN-GRU`, `Ridge Residual`)
4. Hàm huấn luyện Direct/Residual, lưu trọng số & tính thước đo thang giá gốc
5. Huấn luyện đầy đủ lưới Holdout (4 bộ FS × 4 tỷ lệ × 3 chiến lược × 9 mô hình = 432 fits) & lưu trọng số
6. Huấn luyện Walk-Forward căn chỉnh 80/20 (542 phiên, `FS06_WTI_CU`, 9 folds × 5 seeds) & kiểm định DM-HAC
7. Đóng gói `03_nonsa_artifacts.zip` (nạp trực tiếp vào Notebook 04)

## 1. Cấu hình đồng nhất & nạp dữ liệu gốc

In [ ]:
import gc
import glob
import hashlib
import json
import time
import warnings
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.stats import norm
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

OUT = Path("/kaggle/working/03_nonsa_artifacts") if Path("/kaggle/working").exists() else Path("03_nonsa_artifacts")
WEIGHTS_DIR = OUT / "holdout_weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# === THAM SỐ ĐỒNG KHỚP 100% VỚI NOTEBOOK 02 ===
TARGET = "Copper"
LOOKBACK = 22
HORIZON = 1
BATCH_SIZE = 64
SEED = 42
VAL_FRACTION = 0.10
RATIOS = (0.80, 0.90, 0.925, 0.95)
SCHEMES = ("expanding", "rolling_756", "recency_weighted")
ROLLING_WINDOW = 756
RECENCY_HALF_LIFE = 252

# Nhánh Non-SA cố định 65 epochs, LR=1e-3 (không tuning HP)
EPOCHS_HOLDOUT = 65
PATIENCE_HOLDOUT = 9
WF_EPOCHS = 45
WF_PATIENCE = 7
WF_STEP = 63
WF_SEEDS = (41, 42, 43, 44, 45)
WF_FEATURE_SET = "FS06_WTI_CU"
RUN_WALK_FORWARD = True

# 4 bộ đặc trưng có lịch sử giá Copper (bắt buộc cho nhánh Residual)
FEATURE_SETS = {
    "FS01_CU":        ["Copper"],
    "FS02_PAPER_CU":  ["Copper", "WTI", "Gold", "Silver"],
    "FS04_METALS_CU": ["Copper", "Aluminium", "Tin", "Silver", "Gold"],
    "FS06_WTI_CU":    ["Copper", "Aluminium", "Tin", "WTI"],
}

EXPECTED_SHA = "9580ede12d474c8f990556842d551dac176b3c16b2d6a4bffb7dc41dd54bf600"
candidates = (
    glob.glob("/kaggle/input/**/dataset_9_commodities_daily.csv", recursive=True)
    + glob.glob("**/dataset_9_commodities_daily.csv", recursive=True)
)
if not candidates:
    raise FileNotFoundError("Thiếu dataset_9_commodities_daily.csv. Hãy Add Data vào Kaggle.")

DATA_PATH = Path(candidates[0])
DATA_SHA = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert DATA_SHA == EXPECTED_SHA, "SHA-256 dữ liệu không khớp bộ chuẩn."

data = pd.read_csv(DATA_PATH)
data["Date"] = pd.to_datetime(data["Date"], errors="raise")
data = data.sort_values("Date").reset_index(drop=True)
assert len(data) == 2710 and data["Date"].is_unique

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print(f"Sẵn sàng | GPU: {tf.config.list_physical_devices('GPU')} | Feature sets: {list(FEATURE_SETS)}")
print(f"Lưới Holdout Non-SA: {len(FEATURE_SETS)} FS × {len(RATIOS)} splits × {len(SCHEMES)} schemes × 9 models = {len(FEATURE_SETS)*len(RATIOS)*len(SCHEMES)*9} fits")

## 2. Chia tập chống rò rỉ (`make_context`) & trọng số suy giảm (`sample_weights`)

In [ ]:
def reset_keras(seed: int = SEED) -> None:
    tf.keras.backend.clear_session()
    gc.collect()
    np.random.seed(int(seed))
    tf.keras.utils.set_random_seed(int(seed))


def sample_weights(length: int, scheme: str, half_life: int = RECENCY_HALF_LIFE):
    if scheme != "recency_weighted":
        return None
    ages = np.arange(length - 1, -1, -1, dtype=np.float32)
    w = np.exp(-np.log(2.0) * ages / float(half_life))
    return (w / np.mean(w)).astype("float32")


def make_context(frame: pd.DataFrame, ratio: float, scheme: str, feature_set: str, val_fraction: float = VAL_FRACTION) -> dict:
    n = len(frame)
    split = int(n * ratio)
    val_start = int(split * (1.0 - val_fraction))
    fit_start = 0 if scheme != "rolling_756" else max(0, val_start - ROLLING_WINDOW)
    cols = FEATURE_SETS[feature_set]
    assert LOOKBACK + 25 < val_start < split < n

    xsc = MinMaxScaler().fit(frame.iloc[fit_start:val_start][cols])
    ysc = MinMaxScaler().fit(frame.iloc[fit_start:val_start][[TARGET]])
    x_all = xsc.transform(frame[cols]).astype("float32")
    y_all = ysc.transform(frame[[TARGET]]).ravel().astype("float32")

    def build_block(start: int, end: int, restrict_train: bool = False) -> dict:
        indices = np.arange(max(start, LOOKBACK), end, dtype=int)
        if restrict_train:
            indices = indices[indices - LOOKBACK >= fit_start]
        X = np.stack([x_all[t - LOOKBACK : t] for t in indices]).astype("float32")
        return {
            "idx": indices,
            "date": frame["Date"].iloc[indices].to_numpy(),
            "X": X,
            "y": y_all[indices],
            "prev": y_all[indices - 1],
            "y_raw": frame[TARGET].iloc[indices].to_numpy(dtype=np.float64),
            "prev_raw": frame[TARGET].iloc[indices - 1].to_numpy(dtype=np.float64),
        }

    tr = build_block(fit_start, val_start, restrict_train=True)
    va = build_block(val_start, split)
    te = build_block(split, n)
    return {
        "train": tr, "val": va, "test": te,
        "xsc": xsc, "ysc": ysc,
        "features": cols, "feature_set": feature_set,
        "ratio": float(ratio), "scheme": scheme,
        "fit_start": fit_start, "val_start": val_start, "test_start": split,
    }


def make_wf_fold_context(frame: pd.DataFrame, origin: int, end: int, feature_set: str) -> dict:
    val_start = int(origin * (1.0 - VAL_FRACTION))
    cols = FEATURE_SETS[feature_set]
    assert LOOKBACK + 30 < val_start < origin < end <= len(frame)

    xsc = MinMaxScaler().fit(frame.iloc[:val_start][cols])
    ysc = MinMaxScaler().fit(frame.iloc[:val_start][[TARGET]])
    x_all = xsc.transform(frame[cols]).astype("float32")
    y_all = ysc.transform(frame[[TARGET]]).ravel().astype("float32")

    def build_block(first: int, last: int) -> dict:
        ids = np.arange(max(first, LOOKBACK), last, dtype=int)
        return {
            "idx": ids,
            "date": frame["Date"].iloc[ids].to_numpy(),
            "X": np.stack([x_all[t - LOOKBACK : t] for t in ids]).astype("float32"),
            "y": y_all[ids],
            "prev": y_all[ids - 1],
            "y_raw": frame[TARGET].iloc[ids].to_numpy(dtype=np.float64),
            "prev_raw": frame[TARGET].iloc[ids - 1].to_numpy(dtype=np.float64),
        }

    return {
        "train": build_block(LOOKBACK, val_start),
        "val": build_block(val_start, origin),
        "test": build_block(origin, end),
        "xsc": xsc, "ysc": ysc,
        "features": cols, "feature_set": feature_set,
        "ratio": float(origin / len(frame)), "scheme": "expanding",
        "fit_start": 0, "val_start": val_start, "test_start": origin,
    }

## 3. Định nghĩa tách từng cell cho 5 họ mô hình Non-SA (Direct & Residual dùng chung backbone)

### 3.1 Kiến trúc `LSTM` — `(B, 22, F) → LSTM(64) → LSTM(32) → Dense(16) → Dense(1)`

In [ ]:
def build_lstm(input_shape: tuple) -> tf.keras.Model:
    inp = tf.keras.Input(shape=input_shape)
    z = tf.keras.layers.LSTM(64, return_sequences=True)(inp)
    z = tf.keras.layers.Dropout(0.20)(z)
    z = tf.keras.layers.LSTM(32)(z)
    z = tf.keras.layers.Dropout(0.20)(z)
    z = tf.keras.layers.Dense(16, activation="relu")(z)
    out = tf.keras.layers.Dense(1, activation="linear")(z)
    model = tf.keras.Model(inp, out, name="LSTM")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    return model

### 3.2 Kiến trúc `GRU` — `(B, 22, F) → GRU(64) → GRU(32) → Dense(16) → Dense(1)`

In [ ]:
def build_gru(input_shape: tuple) -> tf.keras.Model:
    inp = tf.keras.Input(shape=input_shape)
    z = tf.keras.layers.GRU(64, return_sequences=True)(inp)
    z = tf.keras.layers.Dropout(0.20)(z)
    z = tf.keras.layers.GRU(32)(z)
    z = tf.keras.layers.Dropout(0.20)(z)
    z = tf.keras.layers.Dense(16, activation="relu")(z)
    out = tf.keras.layers.Dense(1, activation="linear")(z)
    model = tf.keras.Model(inp, out, name="GRU")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    return model

### 3.3 Kiến trúc `Causal CNN-LSTM` — `(B, 22, F) → Conv1D(32, k=3, causal) → MaxPool(2) → (B, 11, 32) → LSTM(48→24) → Dense(16→1)`

In [ ]:
def build_cnn_lstm(input_shape: tuple) -> tf.keras.Model:
    inp = tf.keras.Input(shape=input_shape)
    z = tf.keras.layers.Conv1D(32, kernel_size=3, padding="causal", activation="relu")(inp)
    z = tf.keras.layers.MaxPooling1D(pool_size=2)(z)
    z = tf.keras.layers.LSTM(48, return_sequences=True)(z)
    z = tf.keras.layers.Dropout(0.20)(z)
    z = tf.keras.layers.LSTM(24)(z)
    z = tf.keras.layers.Dropout(0.20)(z)
    z = tf.keras.layers.Dense(16, activation="relu")(z)
    out = tf.keras.layers.Dense(1, activation="linear")(z)
    model = tf.keras.Model(inp, out, name="CNN_LSTM")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    return model

### 3.4 Kiến trúc `Causal CNN-GRU` — `(B, 22, F) → Conv1D(32, k=3, causal) → MaxPool(2) → (B, 11, 32) → GRU(48→24) → Dense(16→1)`

In [ ]:
def build_cnn_gru(input_shape: tuple) -> tf.keras.Model:
    inp = tf.keras.Input(shape=input_shape)
    z = tf.keras.layers.Conv1D(32, kernel_size=3, padding="causal", activation="relu")(inp)
    z = tf.keras.layers.MaxPooling1D(pool_size=2)(z)
    z = tf.keras.layers.GRU(48, return_sequences=True)(z)
    z = tf.keras.layers.Dropout(0.20)(z)
    z = tf.keras.layers.GRU(24)(z)
    z = tf.keras.layers.Dropout(0.20)(z)
    z = tf.keras.layers.Dense(16, activation="relu")(z)
    out = tf.keras.layers.Dense(1, activation="linear")(z)
    model = tf.keras.Model(inp, out, name="CNN_GRU")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    return model

### 3.5 Mô hình tuyến tính `Ridge Residual` — duỗi phẳng `(N, 22×F)` → dự báo `Δy_scaled`

In [ ]:
def fit_ridge_residual(ctx: dict, save_dir: Path = None) -> dict:
    """Hồi quy Ridge (alpha=1.0) dự báo phần dư Δy_scaled trên ma trận duỗi phẳng (N, 22*F)."""
    tr, va, te = ctx["train"], ctx["val"], ctx["test"]
    t0 = time.monotonic()
    model = Ridge(alpha=1.0)
    w = sample_weights(len(tr["idx"]), ctx["scheme"])
    model.fit(tr["X"].reshape(len(tr["X"]), -1), tr["y"] - tr["prev"], sample_weight=w)
    fit_sec = float(time.monotonic() - t0)

    val_pred = va["prev"] + model.predict(va["X"].reshape(len(va["X"]), -1))
    test_pred = te["prev"] + model.predict(te["X"].reshape(len(te["X"]), -1))

    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)
        joblib.dump(model, save_dir / "model.joblib")
        joblib.dump(ctx["xsc"], save_dir / "xsc.joblib")
        joblib.dump(ctx["ysc"], save_dir / "ysc.joblib")
        meta = {
            "model": "ridge_residual",
            "target_mode": "residual",
            "feature_set": ctx["feature_set"],
            "features": ctx["features"],
            "ratio": ctx["ratio"],
            "scheme": ctx["scheme"],
            "seed": int(SEED),
            "alpha": 1.0,
        }
        (save_dir / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    return {"val_pred": val_pred, "test_pred": test_pred, "epochs_run": 0, "fit_seconds": fit_sec}


ARCH_BUILDERS = {
    "lstm": build_lstm,
    "gru": build_gru,
    "cnn_lstm": build_cnn_lstm,
    "cnn_gru": build_cnn_gru,
}
NONSA_MODEL_NAMES = [
    f"{arch}_{obj}" for obj in ("direct", "residual") for arch in ("lstm", "gru", "cnn_lstm", "cnn_gru")
] + ["ridge_residual"]
print("Danh sách 9 mô hình Non-SA:", NONSA_MODEL_NAMES)

## 4. Hàm huấn luyện chung Direct/Residual, lưu trọng số & đánh giá thang gốc

In [ ]:
def train_nonsa_model(
    ctx: dict,
    model_name: str,
    seed: int = SEED,
    epochs: int = EPOCHS_HOLDOUT,
    patience: int = PATIENCE_HOLDOUT,
    save_dir: Path = None,
) -> dict:
    if model_name == "ridge_residual":
        return fit_ridge_residual(ctx, save_dir=save_dir)

    is_residual = model_name.endswith("_residual")
    arch = model_name.rsplit("_", 1)[0]
    reset_keras(seed)

    tr, va, te = ctx["train"], ctx["val"], ctx["test"]
    model = ARCH_BUILDERS[arch](tr["X"].shape[1:])
    target_tr = (tr["y"] - tr["prev"]) if is_residual else tr["y"]
    target_va = (va["y"] - va["prev"]) if is_residual else va["y"]
    weights = sample_weights(len(tr["idx"]), ctx["scheme"])

    cb = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True, verbose=0
        )
    ]

    t0 = time.monotonic()
    hist = model.fit(
        tr["X"], target_tr,
        validation_data=(va["X"], target_va),
        sample_weight=weights,
        epochs=epochs,
        batch_size=BATCH_SIZE,
        shuffle=False,
        verbose=0,
        callbacks=cb,
    )
    fit_sec = float(time.monotonic() - t0)

    pv = model.predict(va["X"], verbose=0).ravel()
    pt = model.predict(te["X"], verbose=0).ravel()
    if is_residual:
        pv = va["prev"] + pv
        pt = te["prev"] + pt

    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)
        model.save(save_dir / "model.keras")
        joblib.dump(ctx["xsc"], save_dir / "xsc.joblib")
        joblib.dump(ctx["ysc"], save_dir / "ysc.joblib")
        meta = {
            "model": model_name,
            "target_mode": "residual" if is_residual else "direct",
            "feature_set": ctx["feature_set"],
            "features": ctx["features"],
            "ratio": ctx["ratio"],
            "scheme": ctx["scheme"],
            "seed": int(seed),
            "epochs_run": len(hist.history["loss"]),
        }
        (save_dir / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    del model
    reset_keras(seed)
    return {"val_pred": pv, "test_pred": pt, "epochs_run": len(hist.history["loss"]), "fit_seconds": fit_sec}


def evaluate_predictions(ctx: dict, model_name: str, val_pred: np.ndarray, test_pred: np.ndarray, seed: int = SEED, extras: dict = None):
    ysc = ctx["ysc"]
    te, va = ctx["test"], ctx["val"]
    pred_raw = ysc.inverse_transform(np.asarray(test_pred).reshape(-1, 1)).ravel()
    y_true = te["y_raw"]
    prev_raw = te["prev_raw"]

    mae = float(mean_absolute_error(y_true, pred_raw))
    rmse = float(np.sqrt(mean_squared_error(y_true, pred_raw)))
    mape = float(np.mean(np.abs((y_true - pred_raw) / y_true)) * 100.0)
    r2 = float(r2_score(y_true, pred_raw))

    row = {
        "protocol": "NonSA",
        "model": model_name,
        "feature_set": ctx["feature_set"],
        "scheme": ctx["scheme"],
        "ratio": ctx["ratio"],
        "seed": int(seed),
        "n_test": len(y_true),
        "mae": mae,
        "rmse": rmse,
        "mape_pct": mape,
        "r2": r2,
        "mae_naive": float(mean_absolute_error(y_true, prev_raw)),
        "val_mse_scaled": float(mean_squared_error(va["y"], val_pred)),
    }
    if extras:
        row.update(extras)

    pred_df = pd.DataFrame({
        "Date": te["date"],
        "ratio": ctx["ratio"],
        "scheme": ctx["scheme"],
        "feature_set": ctx["feature_set"],
        "model": model_name,
        "seed": int(seed),
        "actual": y_true,
        "pred": pred_raw,
        "naive": prev_raw,
    })
    return row, pred_df


def dm_hac(loss_diff: np.ndarray, lags: int = 5) -> dict:
    d = np.asarray(loss_diff, dtype=float)
    n = len(d)
    if n < 20 or not np.isfinite(d).all():
        return {"mean_loss_diff": np.nan, "dm_z": np.nan, "p_value": np.nan}
    d_bar = float(d.mean())
    centered = d - d_bar
    lrv = float(np.dot(centered, centered) / n)
    for lag in range(1, min(lags, n - 1) + 1):
        weight = 1.0 - lag / (lags + 1.0)
        lrv += 2.0 * weight * float(np.dot(centered[lag:], centered[:-lag]) / n)
    if lrv <= 0:
        return {"mean_loss_diff": d_bar, "dm_z": np.nan, "p_value": np.nan}
    z = float(d_bar / np.sqrt(lrv / n))
    return {"mean_loss_diff": d_bar, "dm_z": z, "p_value": float(2.0 * norm.sf(abs(z)))}

## 5. Huấn luyện đầy đủ lưới Holdout (4 bộ FS × 4 tỷ lệ × 3 chiến lược × 9 mô hình = 432 fits) & lưu trọng số

In [ ]:
HOLD_RES_CSV = OUT / "nonsa_holdout_results.csv"
HOLD_PRED_CSV = OUT / "nonsa_holdout_predictions.csv"

done_keys = set()
if HOLD_RES_CSV.exists():
    prev_df = pd.read_csv(HOLD_RES_CSV)
    done_keys = set(zip(prev_df["ratio"].round(4), prev_df["feature_set"], prev_df["scheme"], prev_df["model"]))

jobs = [
    (ratio, fs, scheme, m_name)
    for ratio in RATIOS
    for fs in FEATURE_SETS
    for scheme in SCHEMES
    for m_name in NONSA_MODEL_NAMES
]

print(f"Bắt đầu lưới Holdout Non-SA: {len(done_keys)}/{len(jobs)} đã hoàn thành.")
for i, (ratio, fs, scheme, m_name) in enumerate(jobs, 1):
    key = (round(ratio, 4), fs, scheme, m_name)
    if key in done_keys:
        continue

    ctx = make_context(data, ratio, scheme, fs)
    tag = f"r{int(round(ratio*1000))}_{fs}_{scheme}_{m_name}_s{SEED}"
    save_dir = WEIGHTS_DIR / tag

    fit_out = train_nonsa_model(
        ctx, m_name, seed=SEED,
        epochs=EPOCHS_HOLDOUT, patience=PATIENCE_HOLDOUT,
        save_dir=save_dir,
    )
    row, pred_df = evaluate_predictions(
        ctx, m_name, fit_out["val_pred"], fit_out["test_pred"], seed=SEED,
        extras={
            "epochs_run": fit_out["epochs_run"],
            "fit_seconds": fit_out["fit_seconds"],
            "weight_dir": str(save_dir.relative_to(OUT)),
        },
    )
    pd.DataFrame([row]).to_csv(HOLD_RES_CSV, mode="a", header=not HOLD_RES_CSV.exists(), index=False)
    pred_df.to_csv(HOLD_PRED_CSV, mode="a", header=not HOLD_PRED_CSV.exists(), index=False)
    done_keys.add(key)
    print(f"[{i:03}/{len(jobs)}] {ratio:.3f} | {fs:<14} | {scheme:<16} | {m_name:<18} | MAE={row['mae']:.5f} | R2={row['r2']:.4f}", flush=True)

print("HOÀN TẤT LƯỚI HOLDOUT NON-SA & LƯU TRỌNG SỐ.")

## 6. Walk-Forward căn chỉnh 80/20 (542 phiên, `FS06_WTI_CU`, 9 folds × 5 seeds = 369 fits)
- Đồng khớp 100% lịch ngày với Notebook 02; `ridge_residual` là nghiệm giải tích (deterministic) nên chỉ fit 1 lần mỗi fold (`seed=41`).

In [ ]:
WF_FOLD_CSV = OUT / "nonsa_wf_fold_metrics.csv"
WF_PRED_CSV = OUT / "nonsa_wf_predictions.csv"

if RUN_WALK_FORWARD:
    first_idx = int(len(data) * 0.80)
    folds = [(s, min(s + WF_STEP, len(data))) for s in range(first_idx, len(data), WF_STEP)]

    wf_done = set()
    if WF_FOLD_CSV.exists():
        old_wf = pd.read_csv(WF_FOLD_CSV)
        wf_done = set(zip(old_wf["fold_start"].astype(int), old_wf["seed"].astype(int), old_wf["model"]))

    total_wf = sum(1 if m == "ridge_residual" else len(WF_SEEDS) for m in NONSA_MODEL_NAMES) * len(folds)
    print(f"Walk-Forward Non-SA: {len(folds)} folds | Tổng lượt fit: {total_wf}")

    for fi, (f_start, f_end) in enumerate(folds, 1):
        ctx = make_wf_fold_context(data, f_start, f_end, WF_FEATURE_SET)
        for m_name in NONSA_MODEL_NAMES:
            seeds_to_run = (WF_SEEDS[0],) if m_name == "ridge_residual" else WF_SEEDS
            for seed in seeds_to_run:
                if (f_start, seed, m_name) in wf_done:
                    continue
                fit_out = train_nonsa_model(
                    ctx, m_name, seed=seed,
                    epochs=WF_EPOCHS, patience=WF_PATIENCE,
                    save_dir=None,
                )
                row, pred_df = evaluate_predictions(
                    ctx, m_name, fit_out["val_pred"], fit_out["test_pred"], seed=seed,
                    extras={"fold_number": fi, "fold_start": f_start, "fold_end": f_end, "epochs_run": fit_out["epochs_run"]},
                )
                pd.DataFrame([row]).to_csv(WF_FOLD_CSV, mode="a", header=not WF_FOLD_CSV.exists(), index=False)
                pred_df.to_csv(WF_PRED_CSV, mode="a", header=not WF_PRED_CSV.exists(), index=False)
                wf_done.add((f_start, seed, m_name))
                print(f"WF Fold {fi}/{len(folds)} | seed={seed} | {m_name:<18} | MAE={row['mae']:.5f}", flush=True)

    wfp = pd.read_csv(WF_PRED_CSV)
    seed_rows, dm_rows = [], []
    for (m_name, seed), g in wfp.groupby(["model", "seed"]):
        ae = np.abs(g["actual"] - g["pred"])
        an = np.abs(g["actual"] - g["naive"])
        seed_rows.append({
            "model": m_name, "seed": int(seed), "n": len(g),
            "mae": float(ae.mean()),
            "rmse": float(np.sqrt(np.mean((g["actual"] - g["pred"]) ** 2))),
            "mape_pct": float(np.mean(ae / np.abs(g["actual"])) * 100.0),
            "r2": float(r2_score(g["actual"], g["pred"])),
        })
        dm_rows.append({"model": m_name, "seed": int(seed), "n": len(g), **dm_hac(ae - an, lags=5)})

    pd.DataFrame(seed_rows).to_csv(OUT / "nonsa_wf_seed_summary.csv", index=False)
    pd.DataFrame(dm_rows).to_csv(OUT / "nonsa_wf_dm_hac_by_seed.csv", index=False)
    display(pd.DataFrame(seed_rows).groupby("model")[["mae", "rmse", "r2"]].agg(["mean", "std"]).round(5))

## 7. Xuất manifest & đóng gói `03_nonsa_artifacts.zip` (nạp vào Notebook 04)

In [ ]:
manifest = {
    "notebook": "03_Train_NonSA_Baselines_va_WalkForward.ipynb",
    "dataset_sha256": DATA_SHA,
    "lookback": LOOKBACK,
    "horizon": HORIZON,
    "ratios": list(RATIOS),
    "schemes": list(SCHEMES),
    "feature_sets": FEATURE_SETS,
    "models": NONSA_MODEL_NAMES,
    "epochs_holdout": EPOCHS_HOLDOUT,
    "wf_enabled": RUN_WALK_FORWARD,
    "wf_feature_set": WF_FEATURE_SET,
    "wf_seeds": list(WF_SEEDS),
    "wf_step": WF_STEP,
}
(OUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

zip_path = OUT.parent / "03_nonsa_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.rglob("*")):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(OUT))

print(f"HOÀN TẤT ĐÓNG GÓI: {zip_path} ({zip_path.stat().st_size / (1024*1024):.2f} MB)")